In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

More detail you can read here
[https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

In [2]:
docs_index = ['doc 1','doc 2','doc 3']
corpus_df = pd.read_csv("./assets/corpus_tf_idf.csv", delimiter="\r\n", names=['text'])
corpus_df['document'] = docs_index
corpus_df.set_index("document",drop=True)
corpus_df

C:\Users\maruf\AppData\Local\Temp\ipykernel_23556\4011895792.py:2: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  corpus_df = pd.read_csv("./assets/corpus_tf_idf.csv", delimiter="\r\n", names=['text'])


,text,document
0,farras makan nasi,doc 1
1,makan dan minum,doc 2
2,aku dan dia dia atau aku,doc 3


In [3]:
tfidf = TfidfVectorizer(smooth_idf=False)
result = tfidf.fit_transform(corpus_df.text)

In [5]:
# This is the inverse document frequency
# Total dokumen di bagi dengan dokumen yang mengandung pola
for feature,idf_number in zip(tfidf.get_feature_names_out(), tfidf.idf_):
    print(f"{feature} : {idf_number}")

aku : 2.09861228866811
atau : 2.09861228866811
dan : 1.4054651081081644
dia : 2.09861228866811
farras : 2.09861228866811
makan : 1.4054651081081644
minum : 2.09861228866811
nasi : 2.09861228866811


In [6]:
table_tf_idf = pd.DataFrame(result.toarray(),index=[docs_index],columns=tfidf.get_feature_names_out())
table_tf_idf


,aku,atau,dan,dia,farras,makan,minum,nasi
doc 1,0.000000,0.000000,0.000000,0.000000,0.63907,0.427993,0.000000,0.63907
doc 2,0.000000,0.000000,0.486240,0.000000,0.00000,0.486240,0.726044,0.00000
doc 3,0.650651,0.325326,0.217874,0.650651,0.00000,0.000000,0.000000,0.00000


In [ ]:
query = "aku atau farras makan nasi"

rank_result = {}
corpus = list(tfidf.get_feature_names_out())
for doc in docs_index:
    rank_result[doc] = 0
    for term in query.split():
        # Check apakah query yg di cari ada di dalam 
        if term in corpus:
            nTfIdf = table_tf_idf.loc[doc,term].item()
            rank_result[doc] += nTfIdf

# Create df Ranking
result_rangking = pd.DataFrame({"doc":rank_result.keys(),"score":rank_result.values()})
result_rangking.sort_values(by=['score'], ascending=False, inplace=True)
result_merge = result_rangking.merge(corpus_df[['text','document']], left_on="doc", right_on="document",)
result_merge.drop(columns=['document'], inplace=True)
result_merge

,doc,score,text
0,doc 1,1.706134,farras makan nasi
1,doc 3,0.975977,aku dan dia dia atau aku
2,doc 2,0.486240,makan dan minum


## Using Cosine Similiarity

In [66]:
result_consine=[]
query = tfidf.transform(["makan"])
score = cosine_similarity(query, result).flatten()
rank_index_cosine = score.argsort()[::-1]

for a in rank_index_cosine:
    result_consine.append({'doc':corpus_df.loc[a]['document'],'score':score[a],'text':corpus_df.loc[a]['text']})

pd.DataFrame(result_consine)

,doc,score,text
0,doc 2,0.486240,makan dan minum
1,doc 1,0.427993,farras makan nasi
2,doc 3,0.000000,aku dan dia dia atau aku
